In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
from pathlib import Path
import re, gzip, pickle
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

In [4]:
NEWS_RAW_DIR = Path("/content/drive/MyDrive/semantic-drift/news_crawl")
REDDIT_RAW_DIR = Path("/content/drive/MyDrive/semantic-drift/reddit_data/processed/monthly")

SAVE_DIR = Path("/content/drive/MyDrive/semantic-drift/on_demand_contexts")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

print(NEWS_RAW_DIR.exists(), REDDIT_RAW_DIR.exists())

True True


In [4]:
from collections import Counter
import json

def tokenize(text):
    return re.findall(r"\b[a-zA-Z]{4,}\b", text.lower())

def count_news_words():
    counts = Counter()

    for file in sorted(NEWS_RAW_DIR.glob("*.gz")):
        print("Counting news:", file.name)

        with gzip.open(file, "rt", encoding="utf-8", errors="ignore") as f:
            for line in f:
                counts.update(tokenize(line))

    return counts


def count_reddit_words():
    counts = Counter()

    reddit_files = sorted(REDDIT_RAW_DIR.rglob("*.jsonl"))
    print("Reddit files found:", len(reddit_files))

    for file in reddit_files:
        print("Counting reddit:", file.name)

        with open(file, "r", encoding="utf-8", errors="ignore") as f:
            for line in f:
                try:
                    obj = json.loads(line)
                    text = (
                        obj.get("body", "")
                        or obj.get("text", "")
                        or obj.get("selftext", "")
                        or obj.get("title", "")
                    )
                except:
                    text = line

                counts.update(tokenize(text))

    return counts

In [5]:
def extract_news_contexts(word, max_contexts=300):
    pattern = re.compile(rf"\b{re.escape(word)}\b", re.IGNORECASE)
    contexts = []

    for file in sorted(NEWS_RAW_DIR.glob("*.gz")):
        with gzip.open(file, "rt", encoding="utf-8", errors="ignore") as f:
            for line in f:
                if pattern.search(line):
                    contexts.append(line.strip())

                    if len(contexts) >= max_contexts:
                        return pd.DataFrame({"text": contexts})

    return pd.DataFrame({"text": contexts})


def extract_reddit_contexts(word, max_contexts=300):
    pattern = re.compile(rf"\b{re.escape(word)}\b", re.IGNORECASE)
    contexts = []

    for file in sorted(REDDIT_RAW_DIR.rglob("*.jsonl")):
        with open(file, "r", encoding="utf-8", errors="ignore") as f:
            for line in f:
                if pattern.search(line):
                    contexts.append(line.strip())

                    if len(contexts) >= max_contexts:
                        return pd.DataFrame({"text": contexts})

    return pd.DataFrame({"text": contexts})

In [6]:
news_counts = count_news_words()
reddit_counts = count_reddit_words()

candidate_words = list(set(news_counts.keys()) & set(reddit_counts.keys()))

checkpoint_path = SAVE_DIR / "full_counts_checkpoint.pkl"

with open(checkpoint_path, "wb") as f:
    pickle.dump({
        "news_counts": news_counts,
        "reddit_counts": reddit_counts,
        "candidate_words": candidate_words
    }, f)

print("Saved full checkpoint:", checkpoint_path)
print("Candidate words:", len(candidate_words))

Counting news: news.2007.en.shuffled.deduped.gz
Counting news: news.2008.en.shuffled.deduped.gz
Counting news: news.2009.en.shuffled.deduped.gz
Counting news: news.2010.en.shuffled.deduped.gz
Counting news: news.2011.en.shuffled.deduped.gz
Counting news: news.2012.en.shuffled.deduped.gz
Counting news: news.2013.en.shuffled.deduped.gz
Counting news: news.2014.en.shuffled.deduped.gz
Counting news: news.2015.en.shuffled.deduped.gz
Reddit files found: 578
Counting reddit: 2015-01.jsonl
Counting reddit: 2015-02.jsonl
Counting reddit: 2015-03.jsonl
Counting reddit: 2015-04.jsonl
Counting reddit: 2015-05.jsonl
Counting reddit: 2015-06.jsonl
Counting reddit: 2015-07.jsonl
Counting reddit: 2015-08.jsonl
Counting reddit: 2015-09.jsonl
Counting reddit: 2015-10.jsonl
Counting reddit: 2015-11.jsonl
Counting reddit: 2015-12.jsonl
Counting reddit: 2016-01.jsonl
Counting reddit: 2016-02.jsonl
Counting reddit: 2016-03.jsonl
Counting reddit: 2016-04.jsonl
Counting reddit: 2016-05.jsonl
Counting reddit: 

In [7]:
checkpoint_path = SAVE_DIR / "full_counts_checkpoint.pkl"

with open(checkpoint_path, "rb") as f:
    data = pickle.load(f)

news_counts = data["news_counts"]
reddit_counts = data["reddit_counts"]
candidate_words = data["candidate_words"]

print("Loaded full candidate words:", len(candidate_words))

Loaded full candidate words: 945668


In [5]:
checkpoint_path = SAVE_DIR / "full_counts_checkpoint.pkl"

with open(checkpoint_path, "rb") as f:
    data = pickle.load(f)

news_counts = data["news_counts"]
reddit_counts = data["reddit_counts"]
candidate_words = data["candidate_words"]

print("Loaded words:", len(candidate_words))

Loaded words: 945668


In [6]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

def is_good_candidate(word):
    word = word.lower()

    # remove stopwords
    if word in ENGLISH_STOP_WORDS:
        return False

    # basic cleaning
    if len(word) < 5:
        return False
    if not word.isalpha():
        return False

    # frequency constraints (VERY IMPORTANT)
    if news_counts[word] < 100:
        return False
    if reddit_counts[word] < 100:
        return False

    return True

In [7]:
filtered_candidate_words = [
    w for w in candidate_words
    if is_good_candidate(w)
]

print("Filtered words:", len(filtered_candidate_words))
print(filtered_candidate_words[:50])

Filtered words: 81784
['turnabout', 'crack', 'kishore', 'pleaded', 'fewest', 'midlife', 'responders', 'coker', 'popcap', 'greeters', 'collaring', 'firework', 'academy', 'demond', 'witchdoctors', 'cheaters', 'pusher', 'expressionism', 'brookhaven', 'orpington', 'thermostats', 'roadsters', 'luxemburg', 'emerald', 'brokerage', 'waypoint', 'malaki', 'routier', 'superconductor', 'jamon', 'predictability', 'blurred', 'interviewees', 'locomotion', 'turvy', 'farms', 'guzzled', 'disparities', 'dispassionate', 'vacationing', 'mchale', 'cinder', 'pestle', 'digiorno', 'raced', 'sinead', 'crowdsource', 'fudge', 'bookmaker', 'pilgrim']


In [8]:
filtered_candidate_words = sorted(
    filtered_candidate_words,
    key=lambda w: news_counts[w] + reddit_counts[w],
    reverse=True
)

print(filtered_candidate_words[:50])

['people', 'think', 'really', 'deleted', 'question', 'removed', 'years', 'reddit', 'message', 'going', 'right', 'things', 'thing', 'action', 'questions', 'https', 'better', 'actually', 'doesn', 'school', 'pretty', 'person', 'contact', 'probably', 'world', 'compose', 'little', 'subreddit', 'doing', 'friends', 'thought', 'money', 'index', 'concerns', 'getting', 'automatically', 'great', 'having', 'title', 'point', 'taking', 'friend', 'performed', 'moderators', 'family', 'house', 'different', 'makes', 'trying', 'maybe']


In [9]:
filtered_path = SAVE_DIR / "filtered_candidates.pkl"

with open(filtered_path, "wb") as f:
    pickle.dump(filtered_candidate_words, f)

print("Saved filtered candidates:", filtered_path)

Saved filtered candidates: /content/drive/MyDrive/semantic-drift/on_demand_contexts/filtered_candidates.pkl


In [10]:
top500_words = filtered_candidate_words[:500]

with open(SAVE_DIR / "top500_candidates.pkl", "wb") as f:
    pickle.dump(top500_words, f)

print("Saved top 500 words")

Saved top 500 words


In [14]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

BAD_WORDS = set(ENGLISH_STOP_WORDS) | {
    "people","think","really","deleted","question","removed",
    "years","reddit","message","going","right","things","thing",
    "action","questions","https","better","actually","doesn",
    "school","pretty","person","contact","probably","world",
    "compose","little","subreddit","doing","friends","thought",
    "money","index","concerns","getting","automatically","great",
    "having","title","point","taking","friend","performed",
    "moderators","family","house","different","makes","trying",
    "maybe","good","want","need","know","time","said"
}

# 1. Keep only medium-frequency words
def is_good_candidate(word):
    word = word.lower().strip()

    if word in BAD_WORDS:
        return False
    if len(word) < 5:
        return False
    if not word.isalpha():
        return False

    news_freq = news_counts[word]
    reddit_freq = reddit_counts[word]
    total_freq = news_freq + reddit_freq

    # not too rare, not too common
    if news_freq < 100 or reddit_freq < 100:
        return False
    if total_freq > 30000:
        return False

    return True


filtered_candidate_words = [
    w.lower() for w in candidate_words
    if is_good_candidate(w)
]

filtered_candidate_words = sorted(set(filtered_candidate_words))

print("Filtered words:", len(filtered_candidate_words))
print(filtered_candidate_words[:50])

Filtered words: 66778
['aachen', 'aafes', 'aalborg', 'aaliyah', 'aalto', 'aamir', 'aardman', 'aardvark', 'aarhus', 'aaronovitch', 'aarons', 'aaronson', 'aasif', 'ababa', 'abacha', 'aback', 'abacus', 'abagnale', 'abalone', 'abandonments', 'abandons', 'abarth', 'abase', 'abasement', 'abashed', 'abate', 'abated', 'abatement', 'abates', 'abating', 'abattoir', 'abattoirs', 'abaya', 'abayas', 'abbasid', 'abbess', 'abbeys', 'abbie', 'abbot', 'abbots', 'abbotsford', 'abbottabad', 'abbotts', 'abbreviate', 'abbreviated', 'abbreviation', 'abbreviations', 'abbvie', 'abcnews', 'abdel']


In [15]:
def drift_potential_score(word):
    news_freq = news_counts[word]
    reddit_freq = reddit_counts[word]
    total = news_freq + reddit_freq

    # prefer words much more common in Reddit than news, but still frequent enough
    ratio = reddit_freq / (news_freq + 1)

    return ratio * np.log1p(total)

top500_words = sorted(
    filtered_candidate_words,
    key=drift_potential_score,
    reverse=True
)[:500]

print(top500_words[:100])

['trashcan', 'bandaid', 'unsee', 'pansexual', 'fisting', 'spaceballs', 'bidets', 'suprised', 'westworld', 'crackhead', 'respawn', 'mansplaining', 'turds', 'cthulhu', 'gummies', 'awwww', 'neopets', 'cutscenes', 'dickheads', 'poopy', 'majora', 'rugrats', 'misophonia', 'deviantart', 'villian', 'nuking', 'ragnarok', 'whoring', 'sentience', 'shhhh', 'kickass', 'mufasa', 'telekinesis', 'pantera', 'grandpas', 'whatcha', 'everclear', 'jumanji', 'commies', 'karama', 'mulaney', 'hitlers', 'merica', 'teleported', 'trebuchet', 'whataburger', 'aragorn', 'ejaculating', 'roleplaying', 'hearthstone', 'embarassed', 'sidenote', 'abrahamic', 'homeroom', 'deftones', 'obnoxiously', 'waker', 'roomie', 'catan', 'splatoon', 'waitstaff', 'codependent', 'mononoke', 'umbridge', 'uninstalled', 'macros', 'automata', 'slytherin', 'padme', 'cartman', 'inbetween', 'pitbulls', 'teleporting', 'arseholes', 'shills', 'smite', 'sitewide', 'dookie', 'premade', 'baldur', 'pecker', 'creepypasta', 'bitchin', 'tekken', 'katana

In [16]:
def clean_word(word):
    # remove repeated letters like awwww, shhhh
    if re.search(r"(.)\1{2,}", word):
        return False

    # remove common misspellings pattern (optional heuristic)
    if word.endswith(("eeee", "aaaa")):
        return False

    return True

top500_words = [w for w in top500_words if clean_word(w)]

In [17]:
with open(SAVE_DIR / "top500_candidates.pkl", "wb") as f:
    pickle.dump(top500_words, f)

print("Saved FINAL top500 candidates")

Saved FINAL top500 candidates
